# Finding windfarms using SAR

__Description & purpose__: This Notebook is

<style>
.scibtn {
  background-color: #04AA6D;
  color: white;
  font-size: 15px;
  padding: 12px;
  border: none;
  border-radius: 5px;
  cursor: pointer;
}
.howbtn {
  background-color: #0407aaff;
  color: white;
  font-size: 15px;
  padding: 12px;
  border: none;
  border-radius: 5px;
  cursor: pointer;
}
.databtn {
  background-color: #edfd03ff;
  color: black;
  font-size: 15px;
  padding: 12px;
  border: none;
  border-radius: 5px;
  cursor: pointer;
}

</style>

<button class="scibtn">Science</button> &nbsp; <button class="databtn">Satellite</button>

In [ ]:
# Run this cell if pyeodh is not installed, or needs updating
%pip install --upgrade pyeodh
%pip install hvplot shapely ipywidgets rasterio pyproj

* SAR fun

Based on https://gist.github.com/calebrob6/364506877a0ed011bb51a707fbc13ff3 

In [ ]:
# Find data
# Display data
# Use xarray/stackstac
# do some analysis

In [ ]:
# Import the Python API Client
import pyeodh

# Import all other packages
import math
import shapely 
import rasterio

import pandas as pd
#import hvplot.pandas
import matplotlib.pyplot as plt
from pyproj import Transformer

import urllib.request
from PIL import Image
from io import BytesIO

from IPython.display import display
#from ipywidgets import interact



import os

import matplotlib.pyplot as plt
import numpy as np
import rasterio
import rasterio.features
import pystac_client
import planetary_computer as pc
import stackstac
import rtree
import fiona
import fiona.transform
import shapely.geometry

from IPython.display import Image

In [ ]:
# Connect to the Hub
# base_url can be changed to optionally specify a different server, but the default is the production server

client = pyeodh.Client(
    base_url="https://eodatahub.org.uk"
).get_catalog_service()


# Print a list of the collections held in the Resource Catalogue (their id and description).
# As the Resource Catalogue fills and development continues, the number of collections and the richness of their descriptions will increase
for index, collect in enumerate(client.get_collections(), start=1):
    print(f"{index} -- {collect.id}")
    print(f"{collect.description}")

In [ ]:
# The next thing to do is find some open data

# Let's use the CEDA Sentinel 2 ARD
s2ard_cat = client.get_catalog("public/catalogs/ceda-stac-catalogue").get_collection('sentinel2_ard')

print("id: ", s2ard_cat.id)
print("title: ", s2ard_cat.title)
print("description: ", s2ard_cat.description)

dataset https://dap.ceda.ac.uk/neodc/sentinel_ard/data/sentinel_1/2026/03/14/S1A_20260314_30_asc_175827_175852_VVVH_G0_GB_OSGB_RTCK_SpkRL.tif

In [ ]:
bbox = [5.7458, 53.7594, 7.1686, 54.1994]
catalog = pystac_client.Client.open(
    "https://planetarycomputer.microsoft.com/api/stac/v1"
)
search = catalog.search(
    collections=["sentinel-1-rtc"], bbox=bbox, datetime="2022-07-19/2022-08-19"
)
items = search.get_all_items()
print(f"Found {len(items)} items")
item = items[0]

In [ ]:
Image(url=item.assets["rendered_preview"].href)

In [ ]:
ds = stackstac.stack(
    pc.sign(items), bounds_latlon=bbox, epsg=item.properties['proj:epsg'], resolution=10
)
ds

In [ ]:
img = ds[0,0].values

In [ ]:
plt.figure(figsize=(13.5,7.5))
plt.imshow(np.log(img), vmax=-3)
plt.title("VV polarization", fontsize=15)
plt.axis("off")
plt.show()
plt.close()

In [ ]:
plt.figure(figsize=(13.5,7.5))
plt.imshow(img, vmax=0.1)
plt.title("VV polarization", fontsize=15)
plt.axis("off")
plt.show()
plt.close()

In [ ]:
mask = ds > 0.3
mask_nan = ds.notnull()
vals = (mask & mask_nan).sum(axis=(0,1), dtype=float) / mask_nan.sum(axis=(0,1), dtype=float)


In [ ]:
%%time
vals = vals.compute()

In [ ]:
plt.figure(figsize=(13.5,7.5))
plt.imshow(vals, vmin=0, vmax=0.3)
plt.title("Fraction of observations in which a pixel has value > 0.3", fontsize=15)
plt.axis("off")
plt.show()
plt.close()

In [ ]:
%%time

features = []
shapes = []
original_shapes = []
areas = []
idx = rtree.index.Index()
i = 0
num_thrown_away = 0
with rasterio.open("output_threshold.tif") as f:
    t_features = list(rasterio.features.dataset_features(f, with_nodata=True, geographic=True))
    for feature in t_features:
        geom = fiona.transform.transform_geom("epsg:4326", "epsg:3857", feature["geometry"])
        shape = shapely.geometry.shape(geom)
        area = shape.area
        if feature["properties"]["val"] == 1:  # could include an area threshold here
            feature["properties"]["area"] = shape.area
            feature["properties"]["idx"] = i
            features.append(feature)
            areas.append(shape.area)
            shapes.append(shape)
            idx.insert(id=i, coordinates=shape.bounds)
            i += 1
        else:
            num_thrown_away += 1
print(f"Throwing away {num_thrown_away} polygons")

merges = set()
merge_sets = dict()
for i, src_shape in enumerate(shapes):
    if i not in merges:
        to_merge = set([i])
        for j in idx.nearest(src_shape.bounds, num_results=10):
            dst_shape = shapes[j]
            distance = src_shape.distance(dst_shape)
            if i !=j and distance < 100:
                to_merge.add(j)
                merges.add(j)
        if len(to_merge) > 1:
            merge_sets[i] = to_merge

new_features = []
new_shapes = []
for i, shape in enumerate(shapes):
    if i in merge_sets:        
        merge_shapes = [
            shapes[j]
            for j in merge_sets[i]
        ]
        new_shape = shapely.geometry.MultiPolygon(merge_shapes).convex_hull

        geom = shapely.geometry.mapping(new_shape)
        geom = fiona.transform.transform_geom("epsg:3857", "epsg:4326", geom)

        new_feature = features[i]
        new_feature["properties"]["area"] = new_shape.area
        new_feature["geometry"] = geom
        new_features.append(new_feature)
        new_shapes.append(new_shape)
    elif i in merges:
        pass
    else:
        new_features.append(features[i])
        new_shapes.append(shape)

schema = {
    "geometry": "Polygon",
    "properties": {
        "idx": "int",
        "val": "float",
        "filename": "str",
        "area": "float"
    }
}

with fiona.open("output.geojson", "w", driver="GeoJSON", crs="EPSG:4326", schema=schema) as f:
    for feature in new_features:
        f.write(feature)

In [7]:
import xarray as xr
import matplotlib.pyplot as plt

href = "https://dap.ceda.ac.uk/neodc/sentinel_ard/data/sentinel_1/2026/03/14/S1A_20260314_30_asc_175827_175852_VVVH_G0_GB_OSGB_RTCK_SpkRL.tif"

ds = xr.load(href)
ds.plot.imshow(size=10, cmap="Greys_r", robust=True)
plt.show()

# from https://knowledge.dea.ga.gov.au/notebooks/Real_world_examples/Shipping_lane_identification/
#max_vh = ds.vh_dB.max(dim="time")

#max_vh.plot.imshow(size=10, cmap="Greys_r", robust=True)
#plt.show()

ModuleNotFoundError: No module named 'matplotlib'